# Cross-Era Fantasy Cricket: VACE generation (ComfyUI, headless)

Generates 480x832 / 16 fps clips with **Wan2.1-VACE-1.3B** from a **reference photo + control (skeleton) video**.
Runs on **Kaggle (T4, primary)** or **Colab free (backup)**. ComfyUI runs **headless** (API only, no web UI, no tunnels).
fp16 (the T4 has no bf16), SDPA attention, fp8 umt5 text encoder, core ComfyUI nodes only.

> Every output carries a burned-in label "AI-generated - academic project". These are synthetic videos, not real footage.

## Kaggle setup (do once)
1. **Upload inputs as a Dataset**: kaggle.com -> *Datasets* -> *New Dataset* -> upload a folder `cefc-inputs/` with `refs/` (photos), `controls/` (`control_33.mp4` files from `make_control.py`) and `pairs.csv` (columns: `name,ref,control[,prompt]`). Set it **Private**.
2. **New Notebook** -> *File -> Import Notebook* -> upload this file -> *Add Input* -> pick your dataset.
3. Right panel **Settings**: *Accelerator* = **GPU T4 x2** (we only use one), *Internet* = **On** (needs phone verification), *Persistence* = off. Keep the notebook **Private**.
4. Run the first cells interactively until the **smoke test** passes (17 frames).
5. **Save Version -> Save & Run All (Commit)**. Outputs land in `/kaggle/working/outputs/` (see the version's *Output* tab). Download `outputs.zip` and `kaggle_runs.csv`.

`pairs.csv` example:
```
name,ref,control
bradman_batting,refs/bradman.png,controls/batter_control_33.mp4
holding_bowling,refs/holding.png,controls/bowler_control_33.mp4
```

Colab (backup): upload the same `cefc-inputs/` folder to `/content/`, set the runtime to **T4 GPU**, then run all cells. Note Colab's FAQ restricts some generative uses on the free tier; Kaggle is the primary platform.

In [ ]:
# ---- Cell 1: configuration + platform detection ------------------------------------------------
import os, sys, json, time, shutil, subprocess, csv, glob, threading, uuid, urllib.request, urllib.error
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # use ONE GPU only (T4 x2 would otherwise be visible)

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE:
    SCRATCH, OUT_ROOT, INPUT_SEARCH = "/kaggle/tmp", "/kaggle/working", "/kaggle/input"
elif ON_COLAB:
    SCRATCH, OUT_ROOT, INPUT_SEARCH = "/content/scratch", "/content", "/content"      # upload cefc-inputs/ to /content
else:
    raise SystemExit("Run this notebook on Kaggle (primary) or Colab (backup).")
os.makedirs(SCRATCH, exist_ok=True)
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab", "| scratch:", SCRATCH)

# ---- settings (edit here) ----
FRAMES = 33                  # final clip length, must be 4n+1
SMOKE_FRAMES = 17            # quick first test
SEED = 42
WIDTH, HEIGHT, FPS = 480, 832, 16
STEPS, CFG, SHIFT = 20, 5.0, 8.0
PROMPT = ("full-body photo of a cricketer in white flannels and cap holding a wooden cricket bat, "
          "plain sports ground background")
NEGATIVE = ("blurry, low quality, worst quality, static, deformed, extra fingers, bad hands, bad face, "
            "missing bat, extra limbs, fused fingers, watermark, subtitles, text, cluttered background")
LABEL_TEXT = "AI-generated - academic project"      # burned into every output (ASCII: OpenCV fonts lack the en dash)
RUN_SMOKE_ONLY = False       # True = stop after the smoke test

# Model files (Comfy-Org/Wan_2.1_ComfyUI_repackaged)
HF_REPO = "Comfy-Org/Wan_2.1_ComfyUI_repackaged"
UNET_FILE = "wan2.1_vace_1.3B_fp16.safetensors"
TEXT_ENCODER_FILE = "umt5_xxl_fp8_e4m3fn_scaled.safetensors"     # fp8 text encoder (6.7 GB)
VAE_FILE = "wan_2.1_vae.safetensors"
COMFY_DIR = f"{SCRATCH}/ComfyUI"
PORT = 8188
OUT_DIR = f"{OUT_ROOT}/outputs"          # labelled videos (the ones to show)
RAW_DIR = f"{OUT_ROOT}/outputs_raw"      # same videos without the label (used for metrics)
CSV_PATH = f"{OUT_ROOT}/kaggle_runs.csv"
for d in (OUT_DIR, RAW_DIR):
    os.makedirs(d, exist_ok=True)
assert (FRAMES - 1) % 4 == 0 and (SMOKE_FRAMES - 1) % 4 == 0, "frame counts must be 4n+1"

In [ ]:
# ---- Cell 2: install ComfyUI (headless) ----------------------------------------------------------
def sh(cmd, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=True, **kw)

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout)
if not os.path.exists(f"{COMFY_DIR}/main.py"):
    url = "https://github.com/Comfy-Org/ComfyUI"
    try:
        sh(f"git clone --depth 1 --branch v0.46.0 {url} {COMFY_DIR}")      # tag we planned around
    except subprocess.CalledProcessError:
        sh(f"git clone --depth 1 {url} {COMFY_DIR}")                       # tag missing -> latest
# Keep the preinstalled torch/torchvision/torchaudio; install every OTHER requirement (incl. torchsde!).
import re
KEEP_PREINSTALLED = {"torch", "torchvision", "torchaudio"}
def req_name(line):
    m = re.match(r"\s*([A-Za-z0-9_.\-]+)", line)
    return m.group(1).lower() if m else ""
reqs = [l for l in open(f"{COMFY_DIR}/requirements.txt").read().splitlines()
        if l.strip() and not l.strip().startswith("#") and req_name(l) not in KEEP_PREINSTALLED]
open(f"{SCRATCH}/comfy_reqs.txt", "w").write("\n".join(reqs))
sh(f"{sys.executable} -m pip install -q -r {SCRATCH}/comfy_reqs.txt")
sh(f"{sys.executable} -m pip install -q torchsde")      # explicit: ComfyUI's samplers import it
import torchsde  # fail here, not inside ComfyUI, if it is still missing
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# ---- Cell 3: download ONLY the three model files (~11.3 GB) ---------------------------------------
from huggingface_hub import hf_hub_download
MODELS = [("diffusion_models", UNET_FILE), ("text_encoders", TEXT_ENCODER_FILE), ("vae", VAE_FILE)]
for sub, fname in MODELS:
    dest_dir = f"{COMFY_DIR}/models/{sub}"
    os.makedirs(dest_dir, exist_ok=True)
    dest = f"{dest_dir}/{fname}"
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        print("have", fname); continue
    t0 = time.time()
    p = hf_hub_download(HF_REPO, f"split_files/{sub}/{fname}", local_dir=f"{SCRATCH}/hf_dl")
    shutil.move(p, dest)
    print(f"downloaded {fname} ({os.path.getsize(dest) / 1e9:.2f} GB) in {time.time() - t0:.0f} s")
shutil.rmtree(f"{SCRATCH}/hf_dl", ignore_errors=True)
print(subprocess.run(f"df -h {SCRATCH} | tail -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# ---- Cell 4: start ComfyUI HEADLESS (localhost API only: no UI, no tunnel) ------------------------
BASE = f"http://127.0.0.1:{PORT}"
LOG = open(f"{SCRATCH}/comfy.log", "w")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
server = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch",
     "--force-fp16",                      # T4 has no bf16
     "--use-pytorch-cross-attention",     # SDPA (no xformers / flash-attn / sage)
     "--fp32-vae"],                       # Wan VAE in fp32 avoids fp16 black/NaN frames (VAE is small)
    cwd=COMFY_DIR, env=env, stdout=LOG, stderr=subprocess.STDOUT)

def http_json(path, data=None, timeout=60):
    req = urllib.request.Request(BASE + path, data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

for _ in range(180):
    if server.poll() is not None:
        print(open(f"{SCRATCH}/comfy.log").read()[-3000:]); raise RuntimeError("ComfyUI exited; log above")
    try:
        http_json("/system_stats", timeout=3); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not start in 180 s")
print("ComfyUI up:", http_json("/system_stats")["devices"])

In [ ]:
# ---- Cell 5: confirm node names / input names from /object_info ------------------------------------
REQUIRED = {   # class -> inputs we use
    "UNETLoader": ["unet_name", "weight_dtype"], "CLIPLoader": ["clip_name", "type", "device"], "VAELoader": ["vae_name"],
    "CLIPTextEncode": ["text", "clip"], "LoadImage": ["image"], "LoadVideo": ["file"], "GetVideoComponents": ["video"],
    "WanVaceToVideo": ["positive", "negative", "vae", "width", "height", "length", "batch_size", "strength",
                       "control_video", "reference_image"],
    "ModelSamplingSD3": ["model", "shift"],
    "KSampler": ["model", "seed", "steps", "cfg", "sampler_name", "scheduler", "positive", "negative", "latent_image", "denoise"],
    "TrimVideoLatent": ["samples", "trim_amount"], "VAEDecode": ["samples", "vae"],
    "CreateVideo": ["images", "fps"], "SaveVideo": ["video", "filename_prefix", "format", "codec"],
}

def check_object_info(info, required=REQUIRED):
    """Return a list of problems (empty list = all node classes and input names exist)."""
    problems = []
    for cls, inputs in required.items():
        if cls not in info:
            problems.append(f"node class missing: {cls}"); continue
        have = {**info[cls]["input"].get("required", {}), **info[cls]["input"].get("optional", {})}
        for i in inputs:
            if i not in have:
                problems.append(f"{cls}: no input '{i}' (has: {sorted(have)})")
    return problems

OBJ = http_json("/object_info", timeout=120)
problems = check_object_info(OBJ)
if problems:
    print("\n".join(problems)); raise RuntimeError("Node names differ from what this notebook expects: fix build_workflow() / REQUIRED")
print("All", len(REQUIRED), "node classes and their inputs found.")
print("WanVaceToVideo outputs:", OBJ["WanVaceToVideo"]["output_name"])
print("CLIPLoader types include wan:", "wan" in OBJ["CLIPLoader"]["input"]["required"]["type"][0])

In [ ]:
# ---- Cell 6: helpers (workflow, run, label, VRAM monitor, CSV) --------------------------------------
import cv2, numpy as np

def build_workflow(ref_name, ctrl_name, frames, seed, prompt, negative, prefix):
    """API-format workflow: VACE with reference image + skeleton control video."""
    return {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": UNET_FILE, "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENCODER_FILE, "type": "wan", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": VAE_FILE}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["2", 0]}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"text": negative, "clip": ["2", 0]}},
        "6": {"class_type": "LoadImage", "inputs": {"image": ref_name}},
        "7": {"class_type": "LoadVideo", "inputs": {"file": ctrl_name}},
        "8": {"class_type": "GetVideoComponents", "inputs": {"video": ["7", 0]}},
        "9": {"class_type": "WanVaceToVideo", "inputs": {
            "positive": ["4", 0], "negative": ["5", 0], "vae": ["3", 0], "width": WIDTH, "height": HEIGHT,
            "length": frames, "batch_size": 1, "strength": 1.0,
            "control_video": ["8", 0], "reference_image": ["6", 0]}},
        "10": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": SHIFT}},
        "11": {"class_type": "KSampler", "inputs": {
            "model": ["10", 0], "seed": seed, "steps": STEPS, "cfg": CFG, "sampler_name": "uni_pc", "scheduler": "simple",
            "positive": ["9", 0], "negative": ["9", 1], "latent_image": ["9", 2], "denoise": 1.0}},
        "12": {"class_type": "TrimVideoLatent", "inputs": {"samples": ["11", 0], "trim_amount": ["9", 3]}},
        "13": {"class_type": "VAEDecode", "inputs": {"samples": ["12", 0], "vae": ["3", 0]}},
        "14": {"class_type": "CreateVideo", "inputs": {"images": ["13", 0], "fps": float(FPS)}},
        "15": {"class_type": "SaveVideo", "inputs": {"video": ["14", 0], "filename_prefix": prefix, "format": "auto", "codec": "auto"}},
    }

def gpu_used_mb():
    out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i 0", shell=True,
                         capture_output=True, text=True).stdout.strip().splitlines()
    return int(out[0]) if out and out[0].strip().isdigit() else 0

class VramMonitor:
    """Polls nvidia-smi every 0.5 s in a thread; .peak is the max memory.used seen (whole GPU, MB)."""
    def __init__(self): self.peak, self._stop = 0, threading.Event()
    def __enter__(self):
        def loop():
            while not self._stop.is_set():
                self.peak = max(self.peak, gpu_used_mb()); time.sleep(0.5)
        self._t = threading.Thread(target=loop, daemon=True); self._t.start(); return self
    def __exit__(self, *a): self._stop.set(); self._t.join(timeout=3)

def burn_label(src, dst, text=LABEL_TEXT):
    """Copy a video and burn a small label (bottom-left, dark outline) on every frame."""
    cap = cv2.VideoCapture(src); fps = cap.get(cv2.CAP_PROP_FPS) or FPS
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        h, w = f.shape[:2]; scale = max(0.4, w / 1100); x, y = 8, h - 10
        for dx in (-1, 0, 1):          # outline = 8 offset copies of the SAME thin text (a thick stroke drifts out of line)
            for dy in (-1, 0, 1):
                if dx or dy: cv2.putText(f, text, (x + dx, y + dy), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(f, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)
        frames.append(f)
    cap.release()
    assert frames, f"no frames in {src}"
    h, w = frames[0].shape[:2]; tmp = dst + ".tmp.mp4"
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames: vw.write(f)
    vw.release()
    if shutil.which("ffmpeg"):
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "17", dst], check=True)
        os.remove(tmp)
    else:
        os.replace(tmp, dst)
    return len(frames)

def video_looks_ok(path):
    """Cheap sanity check: not black, no NaN-style garbage (mean brightness / contrast of a few frames)."""
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); stats = []
    for i in np.linspace(0, max(0, n - 1), 5).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
        if ok: stats.append((float(f.mean()), float(f.std())))
    cap.release()
    return bool(stats) and all(m > 8 and s > 5 for m, s in stats), stats

CSV_FIELDS = ["timestamp", "name", "frames", "seed", "status", "wall_s", "peak_vram_mb", "baseline_vram_mb", "output", "error"]
def log_row(row):
    new = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, "a", newline="") as fh:
        w = csv.DictWriter(fh, CSV_FIELDS)
        if new: w.writeheader()
        w.writerow({k: row.get(k, "") for k in CSV_FIELDS}); fh.flush()

def run_one(name, ref_path, ctrl_path, frames, prompt=PROMPT, seed=SEED, tag=""):
    """Run one generation, save raw + labelled outputs immediately, append a CSV row. Never raises."""
    uid = uuid.uuid4().hex[:6]
    in_dir = f"{COMFY_DIR}/input"; os.makedirs(in_dir, exist_ok=True)
    ref_name = f"{name}_{uid}_ref{os.path.splitext(ref_path)[1]}"; ctrl_name = f"{name}_{uid}_ctrl.mp4"
    shutil.copy(ref_path, f"{in_dir}/{ref_name}"); shutil.copy(ctrl_path, f"{in_dir}/{ctrl_name}")
    prefix = f"cefc/{name}_{frames}f_{uid}"
    row = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "name": f"{name}{tag}", "frames": frames, "seed": seed,
           "baseline_vram_mb": gpu_used_mb()}
    t0 = time.time()
    try:
        with VramMonitor() as mon:
            res = http_json("/prompt", {"prompt": build_workflow(ref_name, ctrl_name, frames, seed, prompt, NEGATIVE, prefix),
                                        "client_id": uuid.uuid4().hex})
            pid = res["prompt_id"]
            while True:                                           # poll until ComfyUI reports the job in /history
                hist = http_json(f"/history/{pid}")
                if pid in hist: break
                if server.poll() is not None: raise RuntimeError("ComfyUI process died (OOM?)")
                time.sleep(2)
        entry = hist[pid]
        if entry["status"]["status_str"] != "success":
            msgs = [m for m in entry["status"]["messages"] if m[0] == "execution_error"]
            raise RuntimeError(json.dumps(msgs[-1][1])[:800] if msgs else "execution failed")
        files = [im for o in entry["outputs"].values() for im in o.get("images", [])]
        if files:
            f = files[0]; src = os.path.join(COMFY_DIR, "output", f.get("subfolder", ""), f["filename"])
        else:
            # SaveVideo's history key varies by ComfyUI version: fall back to the newest .mp4 in output/cefc/
            # written during THIS run (the mtime check stops us picking up a stale file from an earlier run).
            cands = [p for p in glob.glob(f"{COMFY_DIR}/output/cefc/*.mp4") if os.path.getmtime(p) >= t0 - 5]
            assert cands, "no output file in history and no new .mp4 in output/cefc/"
            src = max(cands, key=os.path.getmtime)
            print(f"[{name}] history had no file entry; using newest output: {os.path.basename(src)}")
        raw = f"{RAW_DIR}/{name}_{frames}f{tag}.mp4"; lab = f"{OUT_DIR}/{name}_{frames}f{tag}.mp4"
        shutil.copy(src, raw); n = burn_label(raw, lab)           # saved immediately, before anything else can fail
        ok, stats = video_looks_ok(raw)
        row.update(status="ok" if ok else "ok_but_dark_or_flat", output=lab, wall_s=round(time.time() - t0, 1), peak_vram_mb=mon.peak)
        print(f"[{name}] {frames}f done in {row['wall_s']} s, peak VRAM {mon.peak} MB, {n} frames -> {lab}" + ("" if ok else "  !! looks black/flat"))
    except Exception as e:
        row.update(status="FAILED", wall_s=round(time.time() - t0, 1), error=str(e)[:500])
        print(f"[{name}] FAILED after {row['wall_s']} s: {e}")
        try: http_json("/interrupt", {})
        except Exception: pass
    log_row(row)
    return row

def find_inputs():
    """Return (root, pairs) where pairs = list of dicts(name, ref, control, prompt). Uses pairs.csv if present."""
    # Kaggle mounts datasets deep (/kaggle/input/datasets/<user>/<slug>/<folder>/) and via symlinks,
    # so search with `find -L` (follows links, no depth limit) instead of a shallow os.walk.
    def find_names(name, typ="f"):
        if os.name == "posix":                                   # Kaggle / Colab
            r = subprocess.run(["find", "-L", INPUT_SEARCH, "-type", typ, "-name", name], capture_output=True, text=True)
            if r.returncode == 0 or r.stdout:
                return sorted(r.stdout.split("\n")) if r.stdout else []
        out = []                                                 # fallback: unlimited-depth walk that follows links
        for dp, dn, fn in os.walk(INPUT_SEARCH, followlinks=True):
            if (typ == "f" and name in fn) or (typ == "d" and name in dn):
                out.append(os.path.join(dp, name))
        return sorted(out)
    hits = [h for h in find_names("pairs.csv") if h]
    if hits:
        root = os.path.dirname(min(hits, key=len))              # shallowest pairs.csv
    else:
        ctrl_dirs = [h for h in find_names("controls", "d") if h and os.path.isdir(os.path.join(os.path.dirname(h), "refs"))]
        root = os.path.dirname(min(ctrl_dirs, key=len)) if ctrl_dirs else None
    assert root, (f"No input dataset found under {INPUT_SEARCH} (need pairs.csv or refs/ + controls/). "
                  f"Top of tree: {subprocess.run(['find', '-L', INPUT_SEARCH, '-maxdepth', '5'], capture_output=True, text=True).stdout[:1500]}")
    if len(hits) > 1: print("NOTE: several pairs.csv found, using", root)
    pairs = []
    if os.path.exists(f"{root}/pairs.csv"):
        for r in csv.DictReader(open(f"{root}/pairs.csv", encoding="utf-8")):
            pairs.append(dict(name=r["name"], ref=os.path.join(root, r["ref"]), control=os.path.join(root, r["control"]),
                              prompt=(r.get("prompt") or PROMPT)))
    else:
        refs = sorted(glob.glob(f"{root}/refs/*.*")); ctrls = sorted(glob.glob(f"{root}/controls/*.mp4"))
        print(f"WARNING: no pairs.csv -> running ALL {len(refs)} refs x {len(ctrls)} controls")
        for r in refs:
            for c in ctrls:
                pairs.append(dict(name=f"{os.path.splitext(os.path.basename(r))[0]}__{os.path.splitext(os.path.basename(c))[0]}", ref=r, control=c, prompt=PROMPT))
    for p in pairs: assert os.path.exists(p["ref"]) and os.path.exists(p["control"]), f"missing file in {p}"
    return root, pairs

INPUT_ROOT, PAIRS = find_inputs()
print(f"Inputs: {INPUT_ROOT}  ->  {len(PAIRS)} pair(s):", [p["name"] for p in PAIRS])

In [ ]:
# ---- Cell 7: SMOKE TEST (17 frames, first pair). Stops the notebook if it fails. -------------------
p0 = PAIRS[0]
smoke = run_one(p0["name"], p0["ref"], p0["control"], SMOKE_FRAMES, prompt=p0["prompt"], tag="_smoke")
if smoke["status"] != "ok":      # FAILED, or a black / flat video (fp16 NaNs), both stop here
    print(open(f"{SCRATCH}/comfy.log").read()[-2500:])
    raise RuntimeError(f"Smoke test failed ({smoke['status']}): {smoke.get('error', '')}. Fix this before the full run.")
print("SMOKE TEST PASSED. Peak VRAM", smoke["peak_vram_mb"], "MB,", smoke["wall_s"], "s for", SMOKE_FRAMES, "frames.")

In [ ]:
# ---- Cell 8: full run: every pair at FRAMES frames. Each output is saved + logged as soon as it finishes ----
if RUN_SMOKE_ONLY:
    print("RUN_SMOKE_ONLY=True -> stopping here")
else:
    results = []
    for i, p in enumerate(PAIRS, 1):
        print(f"--- pair {i}/{len(PAIRS)}: {p['name']}")
        results.append(run_one(p["name"], p["ref"], p["control"], FRAMES, prompt=p["prompt"]))
    print("\n".join(f"{r['name']:30s} {r['status']:20s} {r['wall_s']:>7} s  peak {r['peak_vram_mb']} MB" for r in results))

In [ ]:
# ---- Cell 9: wrap up: shut down the server, zip outputs ----------------------------------------------
try: server.terminate()
except Exception: pass
shutil.make_archive(f"{OUT_ROOT}/outputs", "zip", OUT_DIR)
print(open(CSV_PATH).read() if os.path.exists(CSV_PATH) else "no CSV")
print(sorted(os.listdir(OUT_DIR)))
print("Download: outputs.zip (labelled), outputs_raw/ (unlabelled, for metrics), kaggle_runs.csv")